# 01 – Preprocessing
**Goal:** turn the 84 selected ICU stays into clean, comparable tables with one row per 15 minutes, ready for gap filling (notebook 02) and the anomaly models (notebooks 03–05).

**Why:** the raw numerics files are messy. The rows are shuffled in time, ~85% of the cuff values are copies, sensors sometimes produce impossible values, and every signal is measured at a different rhythm (every second vs. every ~20 minutes). The models need one clean row per fixed time step.

**Input:** the 84 stays selected in notebook 00 (`selected_stays.csv`, decision D4).
**Output:** one table per stay on a 15-minute grid, with a marker for which values were really measured, labelled events, and a train/test split per patient.

**Steps:**
1. Setup and the selected stays
2. Loading one stay (sorted by time, hours since start)
3. Removing the cuff copies
4. Removing impossible values
5. Resampling to a 15-minute grid (D7), keeping track of what was really measured
6. Labelling deterioration events (D8)
7. Train/test split per patient (D6)
8. Scaling on the training patients only
9. Saving the processed data

The reasoning behind each choice is in `decisions.md`.

## 1. Setup and the selected stays

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = os.path.join("..", "dataset", "mimic4wdb")
TICKS_PER_SECOND = 999.56
SIGNALS = ["HR [bpm]", "RR [rpm]", "SpO2 [%]", "NBPs [mmHg]", "NBPd [mmHg]"]
NBP_COLS = ["NBPs [mmHg]", "NBPd [mmHg]", "NBPm [mmHg]"]

selected = pd.read_csv(os.path.join("..", "selected_stays.csv"))
print(len(selected), "selected stays")
selected.head()

86 selected stays


,file,patient
0,p10014354_81739927n.csv.gz,p10014354
1,p10019003_87033314n.csv.gz,p10019003
2,p10039708_83411188n.csv.gz,p10039708
3,p10039708_85583557n.csv.gz,p10039708
4,p10079700_85594648n.csv.gz,p10079700


## 2. Loading one stay
`load_stay` reads one stay with only the columns we need, sorts the rows by time (the files are shuffled) and adds `hours` since the start of the stay (see "Data conventions" in `decisions.md`).

In [2]:
def load_stay(file):
    """Load one stay: our columns only, sorted by time, with hours since start."""
    d = pd.read_csv(os.path.join(DATA_DIR, file),
                    usecols=lambda c: c == "time" or c in SIGNALS or c in NBP_COLS)
    d = d.sort_values("time").reset_index(drop=True)
    d["hours"] = (d["time"] - d["time"].min()) / TICKS_PER_SECOND / 3600
    return d

example = load_stay(selected["file"].iloc[0])
print(example.shape)
print("rows sorted by time:", example["time"].is_monotonic_increasing)
example.head()

(258061, 8)
rows sorted by time: True


,time,HR [bpm],NBPd [mmHg],NBPm [mmHg],NBPs [mmHg],RR [rpm],SpO2 [%],hours
0,33600,NaN,NaN,NaN,NaN,NaN,95.1,0.000000
1,34624,NaN,NaN,NaN,NaN,NaN,95.1,0.000285
2,35648,NaN,NaN,NaN,NaN,NaN,95.1,0.000569
3,36672,NaN,NaN,NaN,NaN,NaN,95.1,0.000854
4,37696,NaN,NaN,NaN,NaN,NaN,95.1,0.001138


## 3. Removing the cuff copies
The monitor re-writes the last cuff value every ~2 minutes (notebook 00, section 9). We keep only real cuff measurements: a cuff value that is identical to the previous cuff row is a copy. We make only the cuff cells of a copy empty (`NaN`), not the whole row, because HR, RR and SpO2 in that row are real.

In [3]:
def remove_cuff_copies(d):
    """Make cuff values empty (NaN) where they repeat the previous cuff row."""
    d = d.copy()
    cols = [c for c in NBP_COLS if c in d.columns]
    cuff = d.loc[d["NBPs [mmHg]"].notna(), cols].fillna(-1)
    is_copy = ~cuff.ne(cuff.shift()).any(axis=1)
    d.loc[cuff.index[is_copy], cols] = np.nan
    return d

before = example["NBPs [mmHg]"].notna().sum()
example = remove_cuff_copies(example)
after = example["NBPs [mmHg]"].notna().sum()
print(f"cuff values: {before} before, {after} after ({100 * (1 - after / before):.0f}% were copies)")

cuff values: 726 before, 149 after (79% were copies)


**Check (first stay):** 726 cuff values before, 149 after → 79% were copies, in line with notebook 00 (section 9).

## 4. Removing impossible values
Sensors sometimes produce impossible values (e.g. a loose finger clip gives SpO2 = 0). Left in, the models would see a sensor error as deterioration. We use wide plausibility ranges, so only clearly impossible values are removed and real extreme values (which we want to detect) stay. As with the copies, we only empty the cell, not the whole row. The ranges are a choice, documented in `decisions.md`.

In [4]:
VALID_RANGES = {
    "HR [bpm]": (20, 250),
    "RR [rpm]": (3, 60),
    "SpO2 [%]": (50, 100),
    "NBPs [mmHg]": (40, 260),
    "NBPd [mmHg]": (20, 180),
}

def remove_impossible(d):
    """Make values outside the plausible range empty (NaN). Returns the table and the counts."""
    d = d.copy()
    removed = {}
    for col, (low, high) in VALID_RANGES.items():
        bad = d[col].notna() & ((d[col] < low) | (d[col] > high))
        removed[col] = int(bad.sum())
        d.loc[bad, col] = np.nan
    bad_bp = d["NBPs [mmHg]"].notna() & d["NBPd [mmHg]"].notna() & (d["NBPs [mmHg]"] <= d["NBPd [mmHg]"])
    removed["NBPs <= NBPd"] = int(bad_bp.sum())
    d.loc[bad_bp, ["NBPs [mmHg]", "NBPd [mmHg]"]] = np.nan
    return d, removed

example, removed = remove_impossible(example)
print("values removed:", removed)

values removed: {'HR [bpm]': 62, 'RR [rpm]': 26, 'SpO2 [%]': 0, 'NBPs [mmHg]': 0, 'NBPd [mmHg]': 0, 'NBPs <= NBPd': 0}


In [5]:
raw = load_stay(selected["file"].iloc[0])
print(raw.loc[(raw["HR [bpm]"] < 20) | (raw["HR [bpm]"] > 250), "HR [bpm]"].value_counts().head())

HR [bpm]
0.0    62
Name: count, dtype: int64


**Check (first stay):** 62 HR and 26 RR values removed (< 0.03% of the rows); no impossible SpO2 or cuff values.

## 5. Resampling to a 15-minute grid (D7)
We cut each stay into 15-minute blocks and average every signal per block, so the models get one row per fixed time step. Blocks without a measurement stay empty (`NaN`); they are filled in notebook 02. For each signal we also keep a `measured` column (True/False), so that later we can always tell real measurements from estimated values.

In [6]:
GRID_MIN = 15

def to_grid(d):
    """Average each signal per 15-minute block; empty blocks stay NaN."""
    block = (d["hours"] * 60 // GRID_MIN).astype(int)
    grid = d.groupby(block)[SIGNALS].mean()
    grid = grid.reindex(range(block.max() + 1))
    grid.index.name = "block"
    for s in SIGNALS:
        grid[s + " measured"] = grid[s].notna()
    grid["hours"] = grid.index * GRID_MIN / 60
    return grid

grid = to_grid(example)
print(grid.shape)
print((100 * grid[[s + " measured" for s in SIGNALS]].mean()).round(1))
grid.head()

(119, 11)
HR [bpm] measured       100.0
RR [rpm] measured       100.0
SpO2 [%] measured       100.0
NBPs [mmHg] measured     63.9
NBPd [mmHg] measured     60.5
dtype: float64


,HR [bpm],RR [rpm],SpO2 [%],NBPs [mmHg],NBPd [mmHg],HR [bpm] measured,RR [rpm] measured,SpO2 [%] measured,NBPs [mmHg] measured,NBPd [mmHg] measured,hours
block,,,,,,,,,,,
0,128.674074,21.423221,99.139423,97.0,60.000000,True,True,True,True,True,0.00
1,119.552901,21.196815,99.025028,NaN,NaN,True,True,True,False,False,0.25
2,118.957859,15.612756,99.099658,NaN,NaN,True,True,True,False,False,0.50
3,114.910125,17.216155,98.632765,108.0,NaN,True,True,True,True,False,0.75
4,114.928246,18.730068,98.982688,83.5,53.333333,True,True,True,True,True,1.00


**Check (first stay):** 119 blocks (~30 h). HR, RR and SpO2 are measured in 100% of the blocks, the cuff in ~60–64% (this patient's cuff measured about every 12 minutes, more often than the typical stay). Some blocks have a systolic but no diastolic value.
Note for D8: this patient starts with HR ≈ 129 bpm, above the event threshold, so the first hours of a stay are not always a stable baseline.